### Modelado y Registro de métricas

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Raíz del proyecto calculada de forma reproducible
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_PATH = BASE_DIR / "data" / "processed" / "train_features.parquet"

df_train = pd.read_parquet(DATA_PATH)
X = df_train.drop(columns=["loan_status"])
y = df_train["loan_status"]

Primeras pruebas tomando 3 modelos (recordar clases desbalanceadas)

In [6]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

# Definir los modelos con parámetros por defecto (solo fijamos la semilla para reproducibilidad)
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "LightGBM": LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1)
}

results = []

# Iterar sobre cada modelo, integrarlo al pipeline y evaluar
for name, model in models.items():
    # El Pipeline empaqueta el escalador y el modelo. 
    # En cada fold de cv=3, ajusta el scaler solo con el train, y transforma el val.
    pipeline = Pipeline([('scaler', RobustScaler().set_output(transform="pandas")), ('classifier', model)])
    
    cv_scores = cross_validate(pipeline, X, y, cv=3, scoring=['precision', 'recall', 'f1'])
    
    # Promediamos las métricas de los 3 folds
    results.append({
        "Model": name,
        "Precision": cv_scores['test_precision'].mean(),
        "Recall": cv_scores['test_recall'].mean(),
        "F1-Score": cv_scores['test_f1'].mean()
    })

# Comparamos los resultados ordenados por F1-Score
df_results = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)
print(df_results.to_markdown(index=False))

| Model               |   Precision |   Recall |   F1-Score |
|:--------------------|------------:|---------:|-----------:|
| Random Forest       |    0.996499 | 1        |   0.998246 |
| LightGBM            |    0.99599  | 0.99749  |   0.996739 |
| Logistic Regression |    0.967412 | 0.924197 |   0.945285 |


Dado por las métricas excelentes se hizo un análisis muy exhaustivo sobre un posible Data Leakage, pero no se encontró. Se entiende que los datos son demasiados fáciles (de jugete) y por eso tienen resultados muy buenos.

Se elige Regresión Logística ya que es el modelo más simple (Navaja de Ockham) y tiene muy buena interpretabilidad. Se pasa ahora a optimizar los hiperparámetros.

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression

# Pipeline base
pipeline_lr = Pipeline([
    ('scaler', RobustScaler()),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])

# Definimos los parámetros a probar (nota el prefijo 'classifier__' para apuntar al modelo)
param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100, 200],  # Valores pequeños = más regularización
    'classifier__penalty': ['l2']              # L2 es el estándar para prevenir sobreajuste
}

# Configuramos y ejecutamos la búsqueda
grid_search = GridSearchCV(
    pipeline_lr, 
    param_grid, 
    cv=3, 
    scoring='f1' 
)

grid_search.fit(X, y)

print(f"Mejor parámetro 'C': {grid_search.best_params_['classifier__C']}")
print(f"Mejor F1-Score en CV: {grid_search.best_score_:.3f}")

# El modelo ya óptimo queda guardado aquí:
best_model_pipeline = grid_search.best_estimator_

Mejor parámetro 'C': 100
Mejor F1-Score en CV: 0.946


#### Interpretación del modelo

Habiendo entrenado el modelo (ya tener loan_model.pkl) se pasa a la interpretación

In [9]:
from pathlib import Path
import joblib
import pandas as pd

# Cargar el modelo guardado
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
MODEL_PATH = BASE_DIR / "models" / "loan_model.pkl"
modelo = joblib.load(MODEL_PATH)

# Nombres de las variables en el mismo orden que entraron
variables = [
    "cibil_score", "annual_payment_burden", "loan_term", 
    "asset_to_loan_ratio", "no_of_dependents", "loan_to_income_ratio"
]

# Extraer los coeficientes de la Regresión Logística
coeficientes = modelo.coef_[0]

# Armar y ordenar la tabla de impacto
df_impacto = pd.DataFrame({"Variable": variables, "Peso_Hacia_Rechazo": coeficientes})
df_ordenado = df_impacto.sort_values(by="Peso_Hacia_Rechazo", ascending=False)

print(df_ordenado.to_markdown(index=False))

| Variable              |   Peso_Hacia_Rechazo |
|:----------------------|---------------------:|
| no_of_dependents      |            0.0983865 |
| asset_to_loan_ratio   |           -0.352308  |
| loan_to_income_ratio  |           -0.853012  |
| loan_term             |           -0.887276  |
| annual_payment_burden |           -1.52745   |
| cibil_score           |           -8.56313   |


El modelo basa su decisión casi enteramente en el historial crediticio (cibil_score). Al tener un peso negativo tan masivo (-8.56), un buen historial arrastra fuertemente la balanza hacia la aprobación del préstamo. El resto de las métricas financieras acompañan esta decisión en menor medida, siendo la cantidad de dependientes (no_of_dependents) el único factor que suma un riesgo muy leve a favor del rechazo, aunque su impacto es mínimo y no logra opacar la importancia fundamental del puntaje de crédito.